# Census Data and TIGER/Line Spatial Join Notebook

This notebook downloads Census ACS data and TIGER/Line census tract boundaries for New York, merges them, calculates population density, and performs a spatial join with training and validation datasets. The final enriched datasets are saved for further analysis.

In [14]:
import os
import pandas as pd
import geopandas as gpd
import requests
from shapely.geometry import Point

## 1. Parameters and ACS Variable Definitions

In [15]:
year = "2022"
state = "36"  # New York FIPS code

# ACS variables (descriptive names mapped to ACS variable codes)
census_vars = {
    'total_population': 'B01003_001E',      # Total Population
    'median_income': 'B19013_001E',          # Median Household Income
    'poverty_count': 'B17001_002E',          # Number below Poverty Level
    'unemployed': 'B23025_005E',             # Unemployed count (for unemployment rate)
    'high_school_grad': 'B15003_017E',       # High School Graduates count (as a proxy)
    'bachelor_degree': 'B15003_022E',        # Bachelor's Degree or higher count
    'under_18': 'B01001_003E',               # (Representative) count of males under 18
    'over_65': 'B01001_020E',                # (Representative) count of males 65+
    'white_alone': 'B02001_002E',            # White alone population
    'black_alone': 'B02001_003E',            # Black or African American alone population
    'housing_units': 'B25001_001E',          # Total Housing Units
    'occupied_units': 'B25002_002E',         # Occupied Housing Units
    'median_home_value': 'B25077_001E',      # Median Home Value
    'gross_rent': 'B25064_001E',             # Median Gross Rent
    'household_size': 'B25010_001E',         # Average Household Size
    'vehicle_ownership': 'B25044_001E',      # Households with a vehicle (total households count)
    'disability': 'B18101_001E',             # Total population reporting a disability
    'foreign_born': 'B05002_013E',           # Foreign born population (representative)
    'labor_force': 'B23025_003E',            # Total Labor Force count
    'crowded_households': 'B25014_005E'      # Crowded Households (as a proxy for household crowding)
}

# List of column names in the order of the API response: first the "NAME", then the above parameters, then state, county, tract.
census_columns = ["NAME"] + list(census_vars.keys()) + ["state", "county", "tract"]

## 2. Download Census ACS Data Using the API

In [20]:
# Note: If you prefer to limit API calls, you could download the data manually as a CSV and read it in.
census_api_key = os.getenv("#", "#")
vars_str = ",".join(census_vars.values())
census_url = (
    f"https://api.census.gov/data/{year}/acs/acs5?"
    f"get=NAME,{vars_str}&for=tract:*&in=state:{state}&in=county:*&key={census_api_key}"
)
print("Census API URL:")


response = requests.get(census_url)
if response.status_code != 200:
    raise Exception(f"Error fetching census data: {response.text}")

# Convert JSON response to DataFrame and assign descriptive column names
data = response.json()
df_census = pd.DataFrame(data[1:], columns=data[0])
# Replace API codes with our descriptive names using the order defined in census_columns
df_census.columns = census_columns

# Convert numeric columns to numbers
for col in list(census_vars.keys()):
    df_census[col] = pd.to_numeric(df_census[col], errors='coerce')

# Create GEOID column by concatenating state, county, and tract codes.
df_census['GEOID'] = df_census['state'] + df_census['county'] + df_census['tract']
print("Census data preview:")
print(df_census.head())

Census API URL:
Census data preview:
                                         NAME  total_population  \
0     Census Tract 1; Albany County; New York              2259   
1  Census Tract 2.01; Albany County; New York              2465   
2  Census Tract 2.02; Albany County; New York              2374   
3  Census Tract 3.01; Albany County; New York              2837   
4  Census Tract 3.02; Albany County; New York              3200   

   median_income  poverty_count  unemployed  high_school_grad  \
0          44547            641         154               352   
1          33688            754         124               286   
2          32585            768          74               263   
3          43214            921         152               392   
4          50875            638         118               389   

   bachelor_degree  under_18  over_65  white_alone  ...  household_size  \
0              173        30        3          750  ...            2.76   
1              404 

## 3. Download and Merge TIGER/Line Census Tract Boundaries

In [21]:
tiger_url = f"https://www2.census.gov/geo/tiger/TIGER{year}/TRACT/tl_{year}_{state}_tract.zip"
print("Downloading TIGER/Line shapefile from:")
print(tiger_url)

# Read the census tract shapefile for New York
df_tract = gpd.read_file(tiger_url)
print("Census tract shapefile preview:")
print(df_tract.head())

# Merge the census data into the tract GeoDataFrame using GEOID
df_tract = df_tract.merge(df_census[['GEOID'] + list(census_vars.keys())], on='GEOID', how='left')
print("After merging census variables into tract data:")
print(df_tract.head())

https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_36_tract.zip
Census tract shapefile preview:
  STATEFP COUNTYFP TRACTCE        GEOID    NAME             NAMELSAD  MTFCC  \
0      36      007  012702  36007012702  127.02  Census Tract 127.02  G5020   
1      36      007  012800  36007012800     128     Census Tract 128  G5020   
2      36      007  012900  36007012900     129     Census Tract 129  G5020   
3      36      007  013000  36007013000     130     Census Tract 130  G5020   
4      36      007  013202  36007013202  132.02  Census Tract 132.02  G5020   

  FUNCSTAT     ALAND  AWATER     INTPTLAT      INTPTLON  \
0        S  65461841  222705  +42.0350532  -075.9055509   
1        S  12342848  259435  +42.1298743  -075.9096569   
2        S  14480163   63649  +42.1522758  -075.9766029   
3        S   9934434  381729  +42.1236499  -076.0002197   
4        S   2446208    3681  +42.1238924  -076.0311921   

                                            geometry  
0  POLYGON (

## 4. Calculate Population Density

In [ ]:
# To accurately calculate area, project to a CRS that uses meters (EPSG:2263 is suitable for New York)
df_tract = df_tract.to_crs(epsg=2263)
df_tract['area_sqkm'] = df_tract['geometry'].area / 1e6  # area in square kilometers
df_tract['population_density'] = df_tract['total_population'] / df_tract['area_sqkm']
print("Census tract data with population density:")
print(df_tract[['GEOID', 'total_population', 'area_sqkm', 'population_density']].head())

## 5. Load Training and Validation Datasets and Spatial Join

In [17]:
# Define file paths for the training and validation CSVs
train_path = '/kaggle/input/uhi-2-updated-eyds/Training_set_combined_with_TempMetrics.csv'
val_path   = '/kaggle/input/uhi-2-updated-eyds/Validation_set_combined_with_TempMetrics.csv'

# Load datasets
df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)
print("Training data preview:")
print(df_train.head())
print("\nValidation data preview:")
print(df_val.head())

# Convert training and validation DataFrames into GeoDataFrames (using Longitude and Latitude columns)
gdf_train = gpd.GeoDataFrame(
    df_train, 
    geometry=gpd.points_from_xy(df_train['Longitude'], df_train['Latitude']),
    crs="EPSG:4326"
)
gdf_val = gpd.GeoDataFrame(
    df_val, 
    geometry=gpd.points_from_xy(df_val['Longitude'], df_val['Latitude']),
    crs="EPSG:4326"
)

# Reproject the training/validation points to match the tract CRS (EPSG:2263)
gdf_train = gdf_train.to_crs(epsg=2263)
gdf_val = gdf_val.to_crs(epsg=2263)

# Spatial join: assign each point the census tract (and hence census variables) that contains it.
gdf_train_joined = gpd.sjoin(gdf_train, df_tract[['GEOID', 'geometry', 'population_density'] + list(census_vars.keys())], how='left', predicate='within')
gdf_val_joined   = gpd.sjoin(gdf_val,   df_tract[['GEOID', 'geometry', 'population_density'] + list(census_vars.keys())], how='left', predicate='within')

print("Training data after spatial join:")
print(gdf_train_joined.head())
print("\nValidation data after spatial join:")
print(gdf_val_joined.head())

Training data preview:
   Longitude   Latitude          datetime  UHI Index  building_count_10m  \
0 -73.909167  40.813107  24/07/2021 15:53   1.030289                   0   
1 -73.909187  40.813045  24/07/2021 15:53   1.030289                   0   
2 -73.909215  40.812978  24/07/2021 15:53   1.023798                   0   
3 -73.909242  40.812908  24/07/2021 15:53   1.023798                   0   
4 -73.909257  40.812845  24/07/2021 15:53   1.021634                   0   

   building_count_20m  building_count_50m  building_count_100m  \
0                   1                   4                    9   
1                   1                   4                   10   
2                   1                   3                    9   
3                   1                   2                    8   
4                   1                   2                    8   

   building_count_150m  building_count_200m  ...  _MaxTemp_ 8  _MinTemp_ 6  \
0                   20                   33  

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


Training data after spatial join:
   Longitude   Latitude          datetime  UHI Index  building_count_10m  \
0 -73.909167  40.813107  24/07/2021 15:53   1.030289                   0   
1 -73.909187  40.813045  24/07/2021 15:53   1.030289                   0   
2 -73.909215  40.812978  24/07/2021 15:53   1.023798                   0   
3 -73.909242  40.812908  24/07/2021 15:53   1.023798                   0   
4 -73.909257  40.812845  24/07/2021 15:53   1.021634                   0   

   building_count_20m  building_count_50m  building_count_100m  \
0                   1                   4                    9   
1                   1                   4                   10   
2                   1                   3                    9   
3                   1                   2                    8   
4                   1                   2                    8   

   building_count_150m  building_count_200m  ...  housing_units  \
0                   20                   33  

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


## 6. Save the Enriched Datasets

In [18]:
output_train = '/kaggle/working/Training_set_Cen.csv'
output_val = '/kaggle/working/Validation_set_Cen.csv'

gdf_train_joined.to_csv(output_train, index=False)
gdf_val_joined.to_csv(output_val, index=False)

print(f"Enriched training dataset saved to: {output_train}")
print(f"Enriched validation dataset saved to: {output_val}")

Enriched training dataset saved to: /kaggle/working/Training_set_Cen.csv
Enriched validation dataset saved to: /kaggle/working/Validation_set_Cen.csv
